In [1]:
import pandas as pd
import numpy as np
import random
from shared import load_data, evaluate_model

In [2]:
train_df, test_df, restaurants_df = load_data()
test_users = test_df['user_id'].unique().tolist()

In [3]:
global_mean = train_df['stars'].mean()
res_stats = train_df.groupby('business_id').agg(
    n_i=('stars', 'count'),
    mu_i=('stars', 'mean')
).reset_index()

In [5]:
def generate_popularity_recs(res_stats_df, global_mean, test_users, lambda_val=50, k=10):
    stats = res_stats_df.copy()

    stats['reg_score'] = (stats['n_i'] * stats['mu_i'] + lambda_val * global_mean) / (stats['n_i'] + lambda_val)
    ranked_items = stats.sort_values('reg_score', ascending=False)['business_id'].tolist()

    top_k_items = ranked_items[:k]
    predictions = {user: top_k_items for user in test_users}

    return predictions

In [10]:
lambda_test = [0, 10, 50, 100, 250, 500]

best_ndcg = 0
best_lambda = 0
best_predictions = None

for l in lambda_test:
    preds = generate_popularity_recs(res_stats, global_mean, test_users, lambda_val=l, k=10)

    metrics = evaluate_model(preds, test_df, k=10)

    print(f"Lambda:{l} | Hit Rate@10: {metrics['Hit@10']:.4f} | NDCG@10: {metrics['NDCG@10']:.4f}")

    if metrics['NDCG@10'] > best_ndcg:
        best_ndcg = metrics['NDCG@10']
        best_lambda = l
        best_predictions = preds

print(f"Optimal Lambda chosen: {best_lambda}")

Lambda:0 | Hit Rate@10: 0.0033 | NDCG@10: 0.0013
Lambda:10 | Hit Rate@10: 0.0148 | NDCG@10: 0.0077
Lambda:50 | Hit Rate@10: 0.0521 | NDCG@10: 0.0221
Lambda:100 | Hit Rate@10: 0.0581 | NDCG@10: 0.0264
Lambda:250 | Hit Rate@10: 0.0619 | NDCG@10: 0.0339
Lambda:500 | Hit Rate@10: 0.0727 | NDCG@10: 0.0378
Optimal Lambda chosen: 500
